# TP Jour 2 — Modèle d'Exécution Spark, Jointures Distribuées & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze**, nettoyer les anomalies, typer fortement les colonnes (dates/timestamps, numériques) et créer vos colonnes métier dérivées.
3. Mettre en œuvre une **jointure distribuée optimisée** avec **Broadcast Hash Join** (`F.broadcast`) sur une table de référence (sans créer de produit cartésien N-M !).
4. Sauvegarder votre table analytique propre dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze & Diagnostic Qualité

### Exercice 1.1 — Profiler les valeurs manquantes, doublons et plages de valeurs
1. Chargez votre (ou vos) table(s) Delta depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` par colonne clé ainsi que le nombre de doublons éventuels sur l'identifiant métier.

In [26]:
# 1. Charger votre table principale depuis `/lakehouse/bronze/transactions`
BRONZE_MAIN_PATH = "/lakehouse/bronze/transactions"
df_bronze = spark.read.format("delta").load(BRONZE_MAIN_PATH)

assert df_bronze is not None, "TODO : Chargez votre table Bronze principale dans `df_bronze`."

# Affichage du schéma et d'un aperçu des 5 premières lignes
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

# 2. Calculer et afficher le nombre de valeurs nulles sur les colonnes principales
cols_to_check = [c for c in df_bronze.columns if not c.startswith("_")]
null_report = df_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cols_to_check])

assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)


root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+--------+-------------------+---------+-------+--------+----------------+-----------+--------------+--------------+-------+----+------+-------------------------+---------------------------------------------------------------+
|id      |date               |client_id|card_id|amount  |use_chip        |merchant_id|merchant_city |merchant_state|zip    |mcc |errors|_ingested_at             |_source_file                        

+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+
|id |date|client_id|card_id|amount|use_chip|merchant_id|merchant_city|merchant_state|zip    |mcc|errors  |
+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+
|0  |0   |0        |0      |0     |0       |0          |0            |1563700       |1652706|0  |13094522|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+



In [28]:
# 1. Charger votre table principale depuis `/lakehouse/bronze/transactions`
BRONZE_MAIN_PATH = "/lakehouse/bronze/transactions"
df_bronze = spark.read.format("delta").load(BRONZE_MAIN_PATH)

assert df_bronze is not None, "TODO : Chargez votre table Bronze principale dans `df_bronze`."

# 2. Calculer et afficher le nombre de valeurs nulles sur les colonnes principales
cols_to_check = [c for c in df_bronze.columns if not c.startswith("_")]
null_report = df_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cols_to_check])
assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)


+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+
|id |date|client_id|card_id|amount|use_chip|merchant_id|merchant_city|merchant_state|zip    |mcc|errors  |
+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+
|0  |0   |0        |0      |0     |0       |0          |0            |1563700       |1652706|0  |13094522|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+



## 2. Nettoyage, Typage Fort & Jointure Optimisée (`BroadcastHashJoin`)

### Rappels d'architecture distribuée :
- **Attention aux jointures N-à-M :** Si vous joignez deux tables de détail partageant une clé non unique, pré-agrégez d'abord à la bonne granularité pour ne pas dupliquer les lignes et fausser vos sommes !
- **Broadcast Hash Join (`F.broadcast(petite_table)`) :** Lorsqu'une table de dimension / nomenclature fait quelques kilo-octets ou méga-octets, la diffuser sur chaque exécuteur supprime totalement l'étape coûteuse de **Shuffle** sur la grande table de faits.
  *(Si votre dataset Kaggle ne comporte qu'un seul fichier plat, construisez ou extrayez une table de référence/catégories dédiée pour pratiquer cette jointure et vérifier le plan physique !)*

### Exercice 2.1 — Nettoyer, enrichir par Broadcast Join et écrire `/lakehouse/silver/<table_silver>`
1. Supprimez les doublons (`dropDuplicates`), filtrez les valeurs techniquement impossibles (ex. durées ou montants négatifs) et convertissez explicitement vos dates (`F.to_timestamp` / `F.to_date`).
2. Créez au moins **2 nouvelles colonnes métier dérivées** utiles pour vos analyses des Jours 3 & 4 (ex. durée écoulée, écart, ratio, tranche horaire, mois `yyyy-MM`, indicateur binaire `0/1`...).
3. Réalisez une jointure avec `F.broadcast(...)` et affichez `.explain("simple")` pour prouver la présence de `BroadcastHashJoin`.
4. Écrivez la table résultante au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [29]:
SILVER_MAIN_PATH = "/lakehouse/silver/transactions"

# 1. Création d'une petite table de référence (nomenclature des modes de paiement) pour le Broadcast Join
ref_payment_modes = spark.createDataFrame(
    [
        ("Swipe Transaction", "Paiement Physique (Piste)", True),
        ("Chip Transaction", "Paiement Physique (Puce)", True),
        ("Online Transaction", "Paiement à Distance (Web)", False),
    ],
    ["use_chip", "payment_label", "is_physical"],
)

# 2. Nettoyage, typage fort et création des colonnes métier dérivées
df_cleaned = (
    df_bronze
    # Suppression d'éventuels doublons sur l'identifiant unique de transaction
    .dropDuplicates(["id"])
    # Conversion de 'amount' : on retire le '$' et on convertit en double
    .withColumn("amount_usd", F.regexp_replace(F.col("amount"), "\\$", "").cast("double"))
    # Filtrage des montants nuls ou négatifs (ex: remboursements ou anomalies)
    .filter(F.col("amount_usd").isNotNull() & (F.col("amount_usd") > 0))
    # Remplacement des NULL logiques (transactions en ligne et absence d'erreur)
    .fillna({"merchant_state": "ONLINE", "errors": "No Error"})
    # Colonnes métier dérivées (temporelles et indicateurs)
    .withColumn("tx_year_month", F.date_format(F.col("date"), "yyyy-MM"))
    .withColumn("tx_hour", F.hour(F.col("date")))
    .withColumn("has_error", F.when(F.col("errors") != "No Error", 1).otherwise(0))
    .drop("amount")  # On supprime l'ancienne colonne texte
)

# 3. Jointure optimisée avec Broadcast Hash Join (pas de Shuffle sur les 13M de lignes !)
df_silver = df_cleaned.join(F.broadcast(ref_payment_modes), on="use_chip", how="left")

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé `df_silver`."

# Preuve physique de l'utilisation de BroadcastHashJoin dans le plan d'exécution
df_silver.explain("simple")

# 4. Sauvegarde au format Delta Lake dans /lakehouse/silver/transactions
(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_MAIN_PATH))

print(
    f"Table Silver écrite dans {SILVER_MAIN_PATH} : {spark.read.format('delta').load(SILVER_MAIN_PATH).count():,} lignes"
)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [use_chip#4697, id#4276, date#4689, client_id#4691, card_id#4693, merchant_id#4699, merchant_city#4701, merchant_state#4581, zip#4705, mcc#4707, errors#4582, _ingested_at#4711, _source_file#4713, amount_usd#4550, tx_year_month#4598, tx_hour#4615, has_error#4633, payment_label#4545, is_physical#4546]
   +- BroadcastHashJoin [use_chip#4697], [use_chip#4544], LeftOuter, BuildRight, false
      :- Project [id#4276, date#4689, client_id#4691, card_id#4693, use_chip#4697, merchant_id#4699, merchant_city#4701, merchant_state#4581, zip#4705, mcc#4707, errors#4582, _ingested_at#4711, _source_file#4713, amount_usd#4550, tx_year_month#4598, tx_hour#4615, CASE WHEN NOT (errors#4582 = No Error) THEN 1 ELSE 0 END AS has_error#4633]
      :  +- Project [id#4276, date#4689, client_id#4691, card_id#4693, use_chip#4697, merchant_id#4699, merchant_city#4701, coalesce(merchant_state#4703, ONLINE) AS merchant_state#4581, zip#4705, mcc#4707,

26/11/26 12:18:39 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


Table Silver écrite dans /lakehouse/silver/transactions : 12,635,227 lignes


---
#### Commentaire explicatif
Analyse de l'exécution et validation du plan physique :
Le pipeline de transformation a permis d'assainir les données brutes issues de la couche Bronze en éliminant les doublons sur l'identifiant unique (id), en convertissant les montants textuels en valeurs numériques flottantes (amount_usd) et en filtrant les transactions nulles ou négatives, aboutissant à un volume propre de 12 635 227 enregistrements. 

Des attributs métier dérivés (granularité mensuelle, tranche horaire et indicateur d'anomalie) ont également été générés pour préparer les analyses ultérieures. L'inspection du plan d'exécution physique confirme la présence de l'opérateur BroadcastHashJoin, démontrant que la table de référence des modes de paiement a été diffusée en mémoire sur l'ensemble des nœuds de calcul, évitant ainsi tout Shuffle coûteux sur la table de faits avant sa persistance au format Delta Lake dans /lakehouse/silver/transactions.

## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [30]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1795175666117,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"7f216542-d26d-4f92-829c-5939244f0abf","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-b1bdaa46-068b-4d8c-b56d-cc2b73332519-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-cd321fe8-3fc0-4e96-ba9c-0b5966fafee4-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-005615e9-4b90-4670-96db-6b03a1a6a70f-c000.snappy.parquet","partition

---
#### Commentaire explicatif
Analyse des mécanismes internes de gouvernance Delta Lake :
La lecture directe du premier fichier de journalisation (00000000000000000000.json) met en évidence l'architecture transactionnelle de Delta Lake : chaque écriture est enregistrée de manière atomique sous forme d'entrées JSON retraçant les métadonnées (metaData), les versions de protocole (protocol) ainsi que la liste exacte des fichiers Parquet actifs (add). 

Par ailleurs, la tentative d'insertion d'un jeu de données au schéma incompatible a été immédiatement interceptée et bloquée par le mécanisme de Schema Enforcement, garantissant l'intégrité structurelle de la table Silver contre toute corruption accidentelle des données.

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [31]:
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)

# 1. Création d'un mini lot de mise à jour (1 ligne existante modifiée + 1 nouvelle ligne fictive)
sample_row = spark.read.format("delta").load(SILVER_MAIN_PATH).limit(1)
updated_row = sample_row.withColumn("amount_usd", F.lit(999.99)).withColumn(
    "errors", F.lit("Manual Audit Correction")
)
new_row = sample_row.withColumn("id", F.lit(-999999)).withColumn("amount_usd", F.lit(50.00))

updates_df = updated_row.unionByName(new_row)

# 2. Exécution du MERGE INTO (Upsert ACID) sur la clé 'id'
(
    target_delta.alias("target")
    .merge(updates_df.alias("source"), "target.id = source.id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# 3. Affichage de l'historique des versions Delta Lake
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, (
    "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"
)

# 4. Preuve du Time Travel : comparaison du nombre de lignes entre la version 0 et la version actuelle (1)
cnt_v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH).count()
cnt_v1 = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
print(f"Time Travel validé -> Lignes Version 0 : {cnt_v0:,} | Lignes Version 1 (après MERGE) : {cnt_v1:,}")


+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      4|2026-11-26 12:42:40.671|    MERGE|{numTargetRowsCopied -> 695025, numTargetRowsDeleted -> 0, numTargetFilesAdde...|
|      3|2026-11-26 12:19:56.469|    WRITE|        {numFiles -> 19, numOutputRows -> 12635227, numOutputBytes -> 289776444}|
|      2|2026-11-20 12:49:26.665| OPTIMIZE|{numRemovedFiles -> 29, numRemovedBytes -> 292928736, p25FileSize -> 28910286...|
|      1|2026-11-20 11:57:32.473|    MERGE|{numTargetRowsCopied -> 694244, numTargetRowsDeleted -> 0, numTargetFilesAdde...|
|      0| 2026-11-20 11:54:26.64|    WRITE|        {numFiles -> 19, numOutputRows -> 12635227, numOutputBytes -> 289776444}|


Time Travel validé -> Lignes Version 0 : 12,635,227 | Lignes Version 1 (après MERGE) : 12,635,228


---
#### Commentaire explicatif

Validation de l'atomicité ACID et de l'auditabilité temporelle :
L'exécution de l'opération MERGE INTO a permis d'appliquer simultanément une mise à jour sur un enregistrement existant et l'insertion d'une nouvelle transaction en une seule transaction atomique, générant ainsi la version 1 dans l'historique de la table (DESCRIBE HISTORY). 

L'interrogation rétroactive via la clause versionAsOf = 0 restitue exactement l'état initial de la table (12 635 227 lignes) face à la version courante (12 635 228 lignes), démontrant la capacité de Time Travel de Delta Lake à assurer une traçabilité complète et une reproductibilité des analyses.